# 1. LIBRARIES

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path to allow imports from functions_final.py
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

# Import functions from functions_final.py
from functions_final import *
from UQpy.distributions import Uniform, JointIndependent

# Set matplotlib parameters for better aesthetics
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False,
                        'figure.dpi': 100
                    })

# 2. LOADING THE CARBONATION MODEL

### 2.1 Load

In [2]:
# Path to the trained model
# name_best_model = r'D:\Documentos\ic_victor\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'
# name_best_model = r'/home/wmpjrufg/Documents/2024-1_victor_hugo_renata_maria/beam_problem_1/model_NeuralNetwork_MLP_fold_4.pkl'
# name_best_model = r'D:\py\2024-1_victor_hugo_renata_maria\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'
name_best_model = r'D:\github\2024-1_victor_hugo_renata_maria\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'

# Load the model
model = joblib.load(name_best_model)
print("Carbonation model loaded successfully!")
print(f"   Expected features: {model.feature_names_in_}")

Carbonation model loaded successfully!
   Expected features: ['CO2 (%)' 'fc (MPa)' 'RH (%)' 'Type of cement' 'Exposure conditions'
 't (years)']


### 2.2 Testing model

In [3]:
mat = {
        'f_ck [MPa]': 30,
        'Type of cement': 2
      }
expo = {
          'Installation year': 1990,
          'Exposure conditions': 2,
          'Relative humidity [%]': 40
        }
geo = {'cover[mm]':30}
load = {}
predictor = CO2Predictor()
beam_with_rh = Beam(geo=geo, mat=mat, load=load, expo=expo)
predictor.set_beam(beam_with_rh)
profile = predictor.carbonation_profile(model_=model, lifetime=150)

In [4]:
profile

,calendar year,t (years),CO2 (%),carbonation depth (mm)
0,1990,0,0.03574,0.589410
1,2000,10,0.03690,10.735841
2,2010,20,0.03893,15.324331
3,2020,30,0.04132,18.963275
4,2030,40,0.04407,21.881557
5,2040,50,0.04718,24.647107
6,2050,60,0.05065,27.325433
7,2060,70,0.05448,29.712950
8,2070,80,0.05867,31.815637
9,2080,90,0.06322,33.933596


In [5]:
carb_depth_mm = predictor.carbonation_depth_at_time(profile, 2025)
carb_depth_mm

20.42241603415687

# 3. DEFINITION OF RANDOM VARIABLES

Defines the probability distributions for the input variables:
- Concrete compressive strength ($f_{ck}$);
- Relative humidity (RH).
- Cover depth (cov).

### 3.1 Design variables

In [6]:
fck_min = 20 # MPa
fck_max = 50 # MPa
rh_min  = 50 # %
rh_max  = 80 # %
cov_min = 2  # mm
cov_max = 6  # mm

### 3.2 Fixed parameters for the analysis

In [7]:
cement_type          = 3
installation_year    = 1990
exposure_conditions  = 2
n_samples            = 50         # Number of design samples. Use 1 for testing one sample
n_latent_samples     = 500        # Number of latent samples per design sample
n_samples_validation = 3          # Number of validation samples
n_lambdas            = 4          # Number of λs to be predicted (λ1, λ2, λ3, λ4)

### 3.3 Samples

In [8]:
# Distributions of random variables
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist   = Uniform(loc=cov_min, scale=cov_max - cov_min)

# Joint distribution
joint = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

# Generate samples
x_pce_rvs = joint.rvs(n_samples)

# Report sample statistics
print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations: {n_samples * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:   {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:    {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover:   {x_pce_rvs[:, 2].min():.3f} - {x_pce_rvs[:, 2].max():.3f} mm (mean: {x_pce_rvs[:, 2].mean():.3f} mm)")

Samples generated successfully!
   Number of design samples: 50
   Number of latent samples per design sample: 500
   Total simulations: 25000

Sample statistics:
   fck:   20.9 - 50.0 MPa (mean: 35.5 MPa)
   RH:    51.0 - 79.9% (mean: 65.4%)
   cover:   2.031 - 5.972 mm (mean: 4.045 mm)


# 4. EVALUATION OF CARBONATION PROGRESS USING GENERALIZED LAMBDA DISTRIBUTION

### 4.1 Step time

In [9]:
times               = np.arange(0, 150, 20)
times               = [10, 20]
complete_model_list = []
pce_dataset         = []

### 4.2 Loop over design samples

In [10]:
print("="*60)
print("BUILDING THE DURABILITY EMULATOR")
print("="*60)

for t in times:
    # ============================================================
    # EMULATOR FUNCTION - CARBONATION DEPTH AND LAMBDAS
    # ============================================================
    print(f'\n{"-"*40}')
    print(f'PROCESSING EMULATOR FOR TIME STEP: {t} years')
    print(f'{"-"*40}')
    df_full, df_unique = emulator_function_time_durability(
                                                                x=x_pce_rvs,
                                                                names_x_variables=["fck", "rh", "cov"],
                                                                carb_model=model,
                                                                cement_type=cement_type,
                                                                installation_year=installation_year,
                                                                exposure_conditions=exposure_conditions,
                                                                time_step=t,
                                                                n_latent_samples=n_latent_samples,
                                                                verbose=False
                                                            )
    # Save the dataset for this time step
    filename = f'dataset_full_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(df_full, f)
    filename = f'dataset_unique_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(df_unique, f)
    print(f'1. The dataset has been saved!')
    # =============================================================
    # BUILDING THE PCE METAMODEL
    # =============================================================
    lambda_cols      = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
    y_pce_rvs        = df_unique[lambda_cols].to_numpy()
    max_degree       = 3
    polynomial_basis = TotalDegreeBasis(joint, max_degree)
    least_squares    = LeastSquareRegression()
    pce_metamodel    = PolynomialChaosExpansion(polynomial_basis=polynomial_basis, regression_method=least_squares)                                                                                                        
    # Train
    pce_metamodel.fit(x_pce_rvs, y_pce_rvs)
    # Save the PCE metamodel for this time step
    filename = f'pce_metamodel_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(pce_metamodel, f)
    print(f'2. PCE training dataset has been saved!')
    # =============================================================
    # VALIDATION THE PCE METAMODEL
    # =============================================================
    x_pce_rvs_val = joint.rvs(n_samples_validation)
    df_full_val, df_unique_val = emulator_function_time_durability(
                                                                        x=x_pce_rvs_val,
                                                                        names_x_variables=["fck", "rh", "cov"],
                                                                        carb_model=model,
                                                                        cement_type=cement_type,
                                                                        installation_year=installation_year,
                                                                        exposure_conditions=exposure_conditions,
                                                                        time_step=t,
                                                                        n_latent_samples=n_latent_samples,
                                                                        verbose=False
                                                                    )
    lambda_cols      = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
    y_pce_val_true   = df_unique_val[lambda_cols].to_numpy()
    y_pce_val_pred   = pce_metamodel.predict(x_pce_rvs_val)
    mse_por_lambda   = []
    r2_por_lambda    = []
    for ii in range(n_lambdas):
        verdade = y_pce_val_true[:, ii]
        predito = y_pce_val_pred[:, ii]
        # MSE computing
        mse = mean_squared_error(verdade, predito)
        mse_por_lambda.append(mse)
        # R² computing
        r2 = r2_score(verdade, predito)
        r2_por_lambda.append(r2)
    statistics_ = pd.DataFrame({'MSE λ1': mse_por_lambda[0], 'MSE λ2': mse_por_lambda[1], 'MSE λ3': mse_por_lambda[2], 'MSE λ4': mse_por_lambda[3], 'R² λ1': r2_por_lambda[0], 'R² λ2': r2_por_lambda[1], 'R² λ3': r2_por_lambda[2], 'R² λ4': r2_por_lambda[3]}, index=[0])
    filename_stats = f'pce_validation_stats_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename_stats, 'wb') as f:
        dill.dump(statistics_, f)
    print(f'3. PCE statistcs has been saved!')

BUILDING THE DURABILITY EMULATOR

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 10 years
----------------------------------------
1. The dataset has been saved!
2. PCE training dataset has been saved!
3. PCE statistcs has been saved!

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 20 years
----------------------------------------
1. The dataset has been saved!
2. PCE training dataset has been saved!
3. PCE statistcs has been saved!


In [ ]:
import dill
filename_stats = f'dataset_unique_10_install_1990_cement_3_exposure_2.pkl'
with open(filename_stats, 'rb') as f:
    statistics_loaded = dill.load(f)
statistics_loaded

,fck,rh,cov,lambda 1,lambda 2,lambda 3,lambda 4
0,35.229311,78.252483,2.455688,-2.839596,1.383110,-0.018056,0.257279
1,36.201033,65.967444,3.308948,-3.366422,1.569342,-0.089293,0.583682
2,41.570906,61.982429,5.345354,-0.767750,1.870496,0.034381,0.059210
3,42.502590,57.414100,5.176410,-1.084499,1.408267,0.470291,0.099090
4,49.993814,71.971927,3.807767,0.226159,1.379563,0.157684,0.282968
5,39.929035,71.245352,2.047031,-3.476182,1.701331,0.028398,0.112628
6,45.047285,79.880360,5.353154,1.911575,2.136920,-0.054891,0.102227
7,39.266614,75.563514,2.333310,-2.576162,1.606686,0.074388,0.216305
8,29.216264,76.733272,5.483933,-1.871168,0.877015,-0.004622,0.496780
9,39.691784,65.124944,3.569717,-2.618087,1.750612,-0.015695,0.145615
